# Implementing a feature store

In this optional notebook, you set up a local Feature Store instance with the fraud detection data set, define and register features, and retrieve them for both training and inference.

For background on Feature Store concepts, see the *Implementing a feature store* chapter in the workshop documentation.

## Install Feature Store

In [ ]:
!pip install feast==0.62.0

## Prepare the data

The fraud detection CSV files have no entity identifier or timestamp. Feature Store requires both: an entity column for feature lookups, and a timestamp for point-in-time joins and materialization ordering.

The following code adds these columns synthetically: `transaction_id` (sequential integer) and `event_timestamp` (relative to current time). This code does not modify the original CSV files in `data/`. Instead, it saves a 10,000-row subset as a new Parquet file.

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

FEAST_REPO = "feast_repo"
DATA_DIR = os.path.join(FEAST_REPO, "data")
SAMPLE_SIZE = 10000

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

df = pd.read_csv('data/train.csv', nrows=SAMPLE_SIZE)
print(f"Loaded {len(df)} rows from data/train.csv (original files are NOT modified)")
print(f"Original columns ({len(df.columns)}): {list(df.columns)}")
print(f"Note: no entity ID, no timestamp — Feast cannot use this data as-is\n")

# Add the two mandatory columns
df['transaction_id'] = range(1, len(df) + 1)

now = pd.Timestamp.now(tz='UTC')
df['event_timestamp'] = now - pd.to_timedelta(range(len(df), 0, -1), unit='s')

cols = ['transaction_id', 'event_timestamp'] + [c for c in df.columns if c not in ['transaction_id', 'event_timestamp']]
df = df[cols]

# Save as a NEW parquet file in feast_repo/ (original data/ folder untouched)
parquet_path = os.path.join(DATA_DIR, "fraud_features.parquet")
df.to_parquet(parquet_path, index=False)

print(f"Saved {len(df)} rows to {parquet_path}")
print(f"New columns ({len(df.columns)}): {list(df.columns)}")
df.head()

## Configure a feature store

Feature Store uses a `feature_store.yaml` file to configure where features are stored. For the purposes of the Fraud Detection tutorial, this notebook uses **local mode** — a file-based registry and SQLite online store — which requires no external infrastructure.

In [ ]:
feature_store_yaml = """project: fraud_detection
provider: local
registry: data/registry.db
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 3
"""

with open(os.path.join(FEAST_REPO, "feature_store.yaml"), 'w') as f:
    f.write(feature_store_yaml)

print("feature_store.yaml written:")
print(feature_store_yaml)

## Define features

Define the core Feature Store objects: Entity (lookup key), FileSource (data location), FeatureView (registered columns), and FeatureService (bundled feature set).

In [ ]:
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource, FeatureService
from feast.types import Float64
from feast.value_type import ValueType

# 1. Entity — the key for feature lookups
transaction = Entity(
    name="transaction",
    join_keys=["transaction_id"],
    value_type=ValueType.INT64,
)

# 2. Data source — points to the Parquet file
fraud_source = FileSource(
    name="fraud_data_source",
    path=os.path.abspath(os.path.join(DATA_DIR, "fraud_features.parquet")),
    timestamp_field="event_timestamp",
)

# 3. Feature view — all data columns from the fraud detection dataset
fraud_features_view = FeatureView(
    name="fraud_features",
    entities=[transaction],
    ttl=timedelta(days=365),
    schema=[
        Field(name="distance_from_home", dtype=Float64),
        Field(name="distance_from_last_transaction", dtype=Float64),
        Field(name="ratio_to_median_purchase_price", dtype=Float64),
        Field(name="repeat_retailer", dtype=Float64),
        Field(name="used_chip", dtype=Float64),
        Field(name="used_pin_number", dtype=Float64),
        Field(name="online_order", dtype=Float64),
        Field(name="fraud", dtype=Float64),
    ],
    source=fraud_source,
    online=True,
)

print(f"Entity: {transaction.name} (join key: {transaction.join_keys})")
print(f"Source: {fraud_source.name}")
print(f"Feature View: {fraud_features_view.name} ({len(fraud_features_view.schema)} features)")

## Define a feature service

A feature service bundles multiple feature views into a named set for consumers to reference.

In [ ]:
fraud_detection_service = FeatureService(
    name="fraud_detection_service",
    features=[fraud_features_view],
)

print(f"Feature Service: {fraud_detection_service.name}")
print(f"  Includes: fraud_features (stored)")

## Apply feature definitions

`store.apply()` validates all definitions, registers them with the Feature Store registry, and sets up the online store infrastructure.

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path=FEAST_REPO)

store.apply([
    transaction,
    fraud_source,
    fraud_features_view,
    fraud_detection_service,
])

print("Feature definitions applied to registry.")
print(f"  Feature views: {[fv.name for fv in store.list_feature_views()]}")
print(f"  Feature services: {[fs.name for fs in store.list_feature_services()]}")

## Materialize to the online store

Materialization copies the latest feature values from the offline store (Parquet) into the online store (SQLite) for low-latency lookups.

In [ ]:
from datetime import datetime

store.materialize_incremental(end_date=datetime.now())
print("\nMaterialization complete. Online store is ready for feature lookups.")

## Retrieve historical features for training

Use `get_historical_features` to retrieve features for a set of entities at specific points in time. Features are returned as named columns rather than positional indices.

In [ ]:
entity_df = pd.DataFrame({
    "transaction_id": list(range(1, 11)),
    "event_timestamp": [pd.Timestamp.now(tz='UTC')] * 10,
})

historical_features = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "fraud_features:distance_from_home",
        "fraud_features:distance_from_last_transaction",
        "fraud_features:ratio_to_median_purchase_price",
        "fraud_features:repeat_retailer",
        "fraud_features:used_chip",
        "fraud_features:used_pin_number",
        "fraud_features:online_order",
        "fraud_features:fraud",
    ],
).to_df()

print(f"Retrieved {len(historical_features)} rows with {len(historical_features.columns)} columns")
historical_features.sort_values('transaction_id').head(10)

## Verify the feature store values

Validate that your feature store returns identical values to the original data.

In [ ]:
feature_cols = [
    'distance_from_home', 'distance_from_last_transaction',
    'ratio_to_median_purchase_price', 'repeat_retailer',
    'used_chip', 'used_pin_number', 'online_order', 'fraud'
]

df_csv = pd.read_csv('data/train.csv', nrows=10)
csv_values = df_csv[feature_cols].values

feast_sorted = historical_features.sort_values('transaction_id').reset_index(drop=True)
feast_values = feast_sorted[feature_cols].values

match = np.allclose(feast_values, csv_values)

print(f"Data integrity check: {'PASSED' if match else 'FAILED'}")
if match:
    print("Feast features are identical to the original CSV data.")
else:
    print("WARNING: Feast features differ from CSV! Check data preparation.")
    for col in feature_cols:
        idx = feature_cols.index(col)
        if not np.allclose(feast_values[:, idx], csv_values[:, idx]):
            print(f"  Mismatch in: {col}")

## Retrieve online features for inference

Use `get_online_features` to look up the latest feature values by entity ID from the online store.

In [ ]:
online_features = store.get_online_features(
    features=[
        "fraud_features:distance_from_last_transaction",
        "fraud_features:ratio_to_median_purchase_price",
        "fraud_features:used_chip",
        "fraud_features:used_pin_number",
        "fraud_features:online_order",
    ],
    entity_rows=[
        {"transaction_id": 1},
        {"transaction_id": 2},
        {"transaction_id": 5},
    ],
).to_dict()

print("Online features for transactions 1, 2, and 5:")
print("(These are the 5 features used by the fraud detection model)\n")
for key, values in online_features.items():
    print(f"  {key}: {values}")

# Use features with the Fraud model

Retrieve features from your feature store, scale them, and run inference with the ONNX model.

**Prerequisite:** Run notebook `1_experiment_train.ipynb` first to create the model and scaler artifacts.

In [ ]:
import pickle

try:
    with open('artifact/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    print("Loaded scaler from artifact/scaler.pkl")
except FileNotFoundError:
    print("Scaler not found. Run notebook 1_experiment_train.ipynb first.")
    scaler = None

In [ ]:
if scaler is not None:
    model_features = [
        "fraud_features:distance_from_last_transaction",
        "fraud_features:ratio_to_median_purchase_price",
        "fraud_features:used_chip",
        "fraud_features:used_pin_number",
        "fraud_features:online_order",
    ]
    model_feature_names = [
        'distance_from_last_transaction', 'ratio_to_median_purchase_price',
        'used_chip', 'used_pin_number', 'online_order'
    ]

    online_result = store.get_online_features(
        features=model_features,
        entity_rows=[{"transaction_id": 1}],
    ).to_dict()

    feature_vector = [[online_result[name][0] for name in model_feature_names]]
    scaled_vector = scaler.transform(feature_vector)

    print("Feature vector from Feature Store online store:")
    for name, val in zip(model_feature_names, feature_vector[0]):
        print(f"  {name}: {val}")
    print(f"\nScaled vector: {scaled_vector[0].tolist()}")

    try:
        import onnxruntime as rt
        sess = rt.InferenceSession("models/fraud/1/model.onnx", providers=rt.get_available_providers())
        input_name = sess.get_inputs()[0].name
        result = sess.run(None, {input_name: scaled_vector.astype(np.float32)})
        prediction = result[0][0]
        print(f"\nONNX model prediction: {'Fraudulent' if prediction == 1 else 'Not fraudulent'}")
        print(f"Raw output: {result[0]}")
    except FileNotFoundError:
        print("\nONNX model not found at models/fraud/1/model.onnx.")
        print("Run notebook 1_experiment_train.ipynb first to create the model.")
    except ImportError:
        print("\nonnxruntime not installed. Install with: pip install onnxruntime")
else:
    print("Skipping — scaler not available.")


## Next steps

This notebook demonstrates core Feature Store functionality with a local setup. For information on using Feature Store in a production environment, see [Configuring Feature Store](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.4/html/working_with_machine_learning_features/configuring_feature_store).